# 47 — HYPOELLIPSE catalog comparison and provenance

Parses the five recovered hypocenter files, verifies their formats against the HYPO71 and
HYPOELLIPSE manuals, establishes the clock offsets between catalogs, compares common events,
and produces the figures for the SRL paper and supplement.

**Inputs** (all relative to `DATA_ROOT`):
- `HYPOCENTERS/moriall.sum` — HYPO71 output, identical to `Pinatubo_all.sum` (catalog MORI)
- `HYPOCENTERS/PINAALL.DAT` — HYPO71 output, partial subset (catalog PINAALL)
- `HYPOCENTERS/GOLD.SUM` — pre-Y2K HYPOELLIPSE archive, 1992–93 processing (catalog GOLD)
- `HYPOCENTERS/b-run.sum` — post-Y2K HYPOELLIPSE archive, 2001 processing (catalog BRUN)
- `HYPOCENTERS/PinatuboRelocs.rtf` — 153 research relocations from PEPI paper (catalog PEPI)

**Outputs**:
- `figures/catalog_maps.pdf/png` — three-panel epicentre map of matched pairs
- `figures/catalog_stats.pdf/png` — comparison statistics (distance, depth, magnitude)
- `products/catalog_inventory.csv` — per-catalog summary table
- `products/clock_offset_table.csv` — documented clock offsets for each catalog

**Manual references**:
- Lahr (1999), USGS OFR 99-23 (HYPOELLIPSE): Section 2.4.1 for summary record format
- Lee & Valdes (1985), USGS OFR 85-749 (HYPO71PC): summary output format

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
from pathlib import Path

DATA_ROOT   = Path('.')          # adjust to point at your archive root
HYPO_DIR    = DATA_ROOT / 'HYPOCENTERS'
FIG_DIR     = DATA_ROOT / 'figures'
PROD_DIR    = DATA_ROOT / 'products'
FIG_DIR.mkdir(exist_ok=True)
PROD_DIR.mkdir(exist_ok=True)

# Clock offsets (hours) to add to raw catalog times to reach UTC.
# Supported by origin-time matching experiments (see paper Section X).
# 'variable' means the offset changes during the eruption sequence:
#   +7 h for May 1 – mid-June 1991 (precursory period)
#    0 h for late-June – August 1991 (post-climactic period)
CLOCK_OFFSETS = {
    'MORI':    'variable',   # +7h May–mid-Jun, 0h late-Jun–Aug
    'PINAALL': 'variable',   # same as MORI (subset of same processing)
    'GOLD':    +7,           # +7h throughout (May 18 – Jun 14 only)
    'BRUN':    0,            # on UTC throughout
    'PEPI':    +7,           # same offset as GOLD for May 30 – Jun 3 window
}

# Match tolerance (seconds) for event association
MATCH_TOL_S   = 10.0   # loose tolerance for initial association
MATCH_TOL_S_STRICT = 2.0  # for statistics plots

# Figure style
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 10,
    'axes.labelsize': 10,
    'axes.titlesize': 10,
})
PINATUBO = (120.35, 15.142)   # approximate summit
MAP_XLIM = (120.10, 120.65)
MAP_YLIM = (14.90, 15.50)

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from datetime import datetime, timedelta
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

## 1  Parsers

### 1.1  HYPO71 summary (MORI / PINAALL)

Format (0-based Python indices after stripping `\r\n`):
- `[0:2]`  yr2 (2-digit year; 91 → 1991)
- `[3:4]`  month (1 digit for May–Aug; would be 2 for Oct–Dec)
- `[4:6]`  day (2 chars)
- `[7:11]` HHMM (4 chars; embedded spaces treated as zeros per FORTRAN I4)
- `[12:18]` seconds (F6.2)
- `[18:20]` lat degrees (I2)
- `[20]`   N/S
- `[21:26]` lat minutes (F5.2)
- `[27:30]` lon degrees (I3)
- `[30]`   E/W
- `[31:36]` lon minutes (F5.2)
- `[37:44]` depth km (F7.2)
- `[44:51]` magnitude (F7.2)

In [ ]:
def _parse_hhmm(s):
    """Parse a 4-char FORTRAN I4 HHMM field where spaces count as zeros."""
    v = int(s.replace(' ', '0'))
    return v // 100, v % 100


def parse_hypo71(path, catalog_id):
    """
    Parse a HYPO71 summary file (moriall.sum or PINAALL.DAT).

    Returns a DataFrame with columns:
        time, lat, lon, dep, mag, catalog
    Times are RAW (before any UTC correction).
    """
    records = []
    path = Path(path)
    for lineno, raw in enumerate(path.read_text(errors='replace').splitlines(), 1):
        line = raw.rstrip('\r\n')
        if len(line) < 44:
            continue
        try:
            yr2  = int(line[0:2])
            mo   = int(line[3:4])          # 1-digit month (May–Aug = 5–8)
            dy   = int(line[4:6])
            hr, mn = _parse_hhmm(line[7:11])
            sc   = float(line[12:18])
            lat_d = int(line[18:20])
            ns    = line[20].upper()
            lat_m = float(line[21:26])
            lon_d = int(line[27:30])
            ew    = line[30].upper()
            lon_m = float(line[31:36])
            dep   = float(line[37:44])
            mag_s = line[44:51].strip()
            mag   = float(mag_s) if mag_s else np.nan
            lat   = lat_d + lat_m / 60.0
            lon   = lon_d + lon_m / 60.0
            if ns == 'S': lat = -lat
            if ew == 'W': lon = -lon
            # Sanity: keep only Philippines-region solutions
            if not (10 < lat < 20 and 115 < lon < 125):
                continue
            sc2 = min(int(sc), 59)
            us  = int((sc - int(sc)) * 1e6)
            t   = datetime(1900 + yr2, mo, dy, hr, mn, sc2, us)
            records.append(dict(time=t, lat=lat, lon=lon, dep=dep, mag=mag,
                                catalog=catalog_id))
        except Exception:
            pass   # skip malformed lines silently
    df = pd.DataFrame(records)
    if not df.empty:
        df['time'] = pd.to_datetime(df['time'])
    print(f"  {catalog_id:8s}: {len(df):5d} events  "
          f"{df.time.min().date() if len(df) else '—'} – "
          f"{df.time.max().date() if len(df) else '—'}")
    return df

### 1.2  HYPOELLIPSE summary: pre-Y2K (GOLD)

Format (Lahr 1999, Section 2.4.1; all columns 0-based, pre-Y2K = post-Y2K shifted **left** 2):
- `[0:2]` yr2, `[2:4]` mo, `[4:6]` dy
- `[6:8]` hr, `[8:10]` mn, `[10:14]` sec×100 (F4.2 stored as integer×100)
- `[14:16]` lat deg (I2), `[16]` N/S, `[17:21]` lat min×100 (F4.2)
- `[21:24]` lon deg (I3), `[24]` E/W, `[25:29]` lon min×100 (F4.2)
- `[29:34]` depth km (F5.2 — **two implied decimal places**)
- `[34:36]` preferred magnitude (F2.1)
- `[79]` `/` summary record marker (column 81 in 1-based)
- `[85:87]` processing month (I2), `[87:89]` processing year 2-digit (I2)

In [ ]:
def parse_hypoellipse_preY2K(path, catalog_id):
    """
    Parse a pre-Y2K HYPOELLIPSE summary file (GOLD.SUM).
    Returns DataFrame with columns: time, lat, lon, dep, mag,
                                     proc_month, proc_year, catalog.
    Times are RAW.
    """
    records = []
    path = Path(path)
    for raw in path.read_text(errors='replace').splitlines():
        line = raw.rstrip('\r\n')
        if len(line) < 80:
            continue
        # Only summary records (marker at col 79, 0-based)
        if line[79] not in ('/', '\\'):
            continue
        try:
            yr2  = int(line[0:2])
            mo   = int(line[2:4])
            dy   = int(line[4:6])
            hr   = int(line[6:8].strip() or 0)
            mn   = int(line[8:10].strip() or 0)
            sc   = float(line[10:14].strip() or 0) / 100.0
            lat_d = line[14:16].strip()
            ns    = line[16]
            lat_m = line[17:21].strip()
            lon_d = line[21:24].strip()
            ew    = line[24]
            lon_m = line[25:29].strip()
            dep_s = line[29:34].strip()
            mag_s = line[34:36].strip()
            # Processing date (cols 85-88, 0-based)
            pm_s  = line[85:87].strip() if len(line) > 88 else ''
            py_s  = line[87:89].strip() if len(line) > 89 else ''
            if not lat_d or not lon_d or not lat_d.lstrip('-').isdigit():
                continue
            lat   = float(lat_d) + float(lat_m) / 100.0 / 60.0
            lon   = float(lon_d) + float(lon_m) / 100.0 / 60.0
            if ns.upper() == 'S': lat = -lat
            if ew.upper() == 'W': lon = -lon
            if not (10 < lat < 20 and 115 < lon < 125):
                continue
            # F5.2: two implied decimals
            dep  = float(dep_s) / 100.0 if dep_s and dep_s.lstrip('-').isdigit() else np.nan
            # F2.1: one implied decimal
            mag  = float(mag_s) / 10.0  if mag_s and mag_s.isdigit()             else np.nan
            pm   = int(pm_s) if pm_s.isdigit() else np.nan
            py   = int(py_s) if py_s.isdigit() else np.nan
            sc2  = min(int(sc), 59)
            us   = int((sc - int(sc)) * 1e6)
            t    = datetime(1900 + yr2, mo, dy, hr, mn, sc2, us)
            records.append(dict(time=t, lat=lat, lon=lon, dep=dep, mag=mag,
                                proc_month=pm, proc_year=py, catalog=catalog_id))
        except Exception:
            pass
    df = pd.DataFrame(records)
    if not df.empty:
        df['time'] = pd.to_datetime(df['time'])
    print(f"  {catalog_id:8s}: {len(df):5d} events  "
          f"{df.time.min().date() if len(df) else '—'} – "
          f"{df.time.max().date() if len(df) else '—'}")
    return df

### 1.3  HYPOELLIPSE summary: post-Y2K (BRUN / TESTALL)

Same as pre-Y2K but all column indices shifted **right by 2** (4-digit year in cols 0–3).
- `[0:4]` yr4, `[4:6]` mo, `[6:8]` dy
- `[8:10]` hr, `[10:12]` mn, `[12:16]` sec×100
- `[16:18]` lat deg, `[18]` N/S, `[19:23]` lat min×100
- `[23:26]` lon deg, `[26]` E/W, `[27:31]` lon min×100
- `[31:36]` depth km (F5.2)
- `[36:38]` preferred magnitude (F2.1)
- `[81]` `/` summary record marker (column 83 in 1-based)
- `[87:89]` processing month, `[89:91]` processing year 2-digit

In [ ]:
def parse_hypoellipse_postY2K(path, catalog_id):
    """
    Parse a post-Y2K HYPOELLIPSE summary file (b-run.sum, TESTALL.SUM).
    Returns DataFrame with columns: time, lat, lon, dep, mag,
                                     proc_month, proc_year, catalog.
    Times are RAW.
    """
    records = []
    path = Path(path)
    for raw in path.read_text(errors='replace').splitlines():
        line = raw.rstrip('\r\n')
        if len(line) < 82:
            continue
        # Only primary summary records
        if line[81] not in ('/', '\\'):
            continue
        try:
            yr   = int(line[0:4])
            mo   = int(line[4:6])
            dy   = int(line[6:8])
            hr   = int(line[8:10].strip() or 0)
            mn   = int(line[10:12].strip() or 0)
            sc   = float(line[12:16].strip() or 0) / 100.0
            lat_d = line[16:18].strip()
            ns    = line[18]
            lat_m = line[19:23].strip()
            lon_d = line[23:26].strip()
            ew    = line[26]
            lon_m = line[27:31].strip()
            dep_s = line[31:36].strip()
            mag_s = line[36:38].strip()
            pm_s  = line[87:89].strip() if len(line) > 90 else ''
            py_s  = line[89:91].strip() if len(line) > 91 else ''
            if not lat_d or not lon_d or not lat_d.lstrip('-').isdigit():
                continue
            lat  = float(lat_d) + float(lat_m) / 100.0 / 60.0
            lon  = float(lon_d) + float(lon_m) / 100.0 / 60.0
            if ns.upper() == 'S': lat = -lat
            if ew.upper() == 'W': lon = -lon
            if not (10 < lat < 20 and 115 < lon < 125):
                continue
            dep  = float(dep_s) / 100.0 if dep_s and dep_s.lstrip('-').isdigit() else np.nan
            mag  = float(mag_s) / 10.0  if mag_s and mag_s.isdigit()             else np.nan
            pm   = int(pm_s) if pm_s.isdigit() else np.nan
            py   = int(py_s) if py_s.isdigit() else np.nan
            sc2  = min(int(sc), 59)
            us   = int((sc - int(sc)) * 1e6)
            t    = datetime(yr, mo, dy, hr, mn, sc2, us)
            records.append(dict(time=t, lat=lat, lon=lon, dep=dep, mag=mag,
                                proc_month=pm, proc_year=py, catalog=catalog_id))
        except Exception:
            pass
    df = pd.DataFrame(records)
    if not df.empty:
        df['time'] = pd.to_datetime(df['time'])
    print(f"  {catalog_id:8s}: {len(df):5d} events  "
          f"{df.time.min().date() if len(df) else '—'} – "
          f"{df.time.max().date() if len(df) else '—'}")
    return df

## 2  Load all catalogs

In [ ]:
print("Loading catalogs...")
mori  = parse_hypo71(HYPO_DIR / 'moriall.sum',   'MORI')
pina  = parse_hypo71(HYPO_DIR / 'PINAALL.DAT',   'PINAALL')
gold  = parse_hypoellipse_preY2K(HYPO_DIR / 'GOLD.SUM', 'GOLD')
brun  = parse_hypoellipse_postY2K(HYPO_DIR / 'b-run.sum', 'BRUN')

# Confirm byte-identity of moriall.sum and Pinatubo_all.sum
import hashlib
def md5(p):
    return hashlib.md5(Path(p).read_bytes()).hexdigest()

h1 = md5(HYPO_DIR / 'moriall.sum')
h2 = md5(HYPO_DIR / 'Pinatubo_all.sum')
print(f"\nmoriall.sum == Pinatubo_all.sum: {h1 == h2} (MD5 {h1[:8]}…)")

# Catalog summary table
rows = []
for nm, df in [('MORI (moriall.sum)', mori), ('PINAALL (PINAALL.DAT)', pina),
               ('GOLD (GOLD.SUM)', gold),    ('BRUN (b-run.sum)', brun)]:
    rows.append({
        'Catalog': nm,
        'Events': len(df),
        'Start': df.time.min().date() if len(df) else None,
        'End':   df.time.max().date() if len(df) else None,
        'Dep_med_km': round(df.dep.median(), 1),
        'Mag_med': round(float(np.nanmedian(df.mag.dropna())), 2) if len(df.mag.dropna()) else None,
        'Clock_offset_h': str(CLOCK_OFFSETS.get(nm.split()[0], '?')),
    })
inv = pd.DataFrame(rows)
print("\nCatalog inventory:")
display(inv)
inv.to_csv(PROD_DIR / 'catalog_inventory.csv', index=False)

## 3  Clock offset analysis

We test whether GOLD and MORI (May–June) match BRUN better with or without a +7 h shift,
and confirm that MORI (July–August) matches BRUN at zero offset.

In [ ]:
def count_matches(A, B, offset_h=0, tol_s=5.0):
    """Count how many events in A have a counterpart in B within tol_s seconds
    after adding offset_h hours to A's times."""
    B_ms = B.time.values.astype('datetime64[ms]').astype(np.int64)
    offset = int(offset_h * 3600 * 1000)   # ms
    tol_ms = int(tol_s * 1000)
    n = 0
    for t in A.time.values.astype('datetime64[ms]').astype(np.int64):
        if np.abs(B_ms - (t + offset)).min() <= tol_ms:
            n += 1
    return n


# Scan offsets −10 to +10 hours for GOLD vs BRUN
print("GOLD vs BRUN — scanning integer-hour offsets (−10 to +10):")
results = []
for h in range(-10, 11):
    n = count_matches(gold, brun, offset_h=h, tol_s=5.0)
    results.append((h, n))
    if n > 50:
        print(f"  offset={h:+3d} h → {n:4d} / {len(gold)} GOLD events matched")

best_h = max(results, key=lambda x: x[1])[0]
print(f"\nBest offset for GOLD→BRUN: {best_h:+d} h")

# MORI May–June vs BRUN
mori_mj = mori[mori.time.dt.month.isin([5, 6])]
mori_ja = mori[mori.time.dt.month.isin([7, 8])]
print(f"\nMORI (May–Jun) vs BRUN — +7h: {count_matches(mori_mj, brun, +7, 5.0)} / {len(mori_mj)}")
print(f"MORI (May–Jun) vs BRUN — 0h:  {count_matches(mori_mj, brun,  0, 5.0)} / {len(mori_mj)}")
print(f"MORI (Jul–Aug) vs BRUN — 0h:  {count_matches(mori_ja, brun,  0, 5.0)} / {len(mori_ja)}")
print(f"MORI (Jul–Aug) vs BRUN — +7h: {count_matches(mori_ja, brun, +7, 5.0)} / {len(mori_ja)}")

# Document the offsets
clock_rows = [
    {'Catalog': 'MORI', 'Period': 'May 1 – ~Jun 15', 'Raw_clock': 'WILLIE wall clock (PHT+CA)',
     'UTC_offset_h': +7, 'Confidence': 'high', 'Evidence': 'origin-time matching vs BRUN'},
    {'Catalog': 'MORI', 'Period': 'Jun 29 – Aug 19',  'Raw_clock': 'WILLIE wall clock (UTC)',
     'UTC_offset_h':  0, 'Confidence': 'high', 'Evidence': 'origin-time matching vs BRUN'},
    {'Catalog': 'GOLD', 'Period': 'May 18 – Jun 14', 'Raw_clock': 'unknown (PHT+CA?)',
     'UTC_offset_h': +7, 'Confidence': 'high', 'Evidence': 'origin-time matching vs BRUN'},
    {'Catalog': 'BRUN', 'Period': 'May 1 – Aug 7',   'Raw_clock': 'UTC',
     'UTC_offset_h':  0, 'Confidence': 'high', 'Evidence': 'reference catalog'},
]
clk = pd.DataFrame(clock_rows)
display(clk)
clk.to_csv(PROD_DIR / 'clock_offset_table.csv', index=False)

## 4  Event matching and difference statistics

In [ ]:
def match_catalogs(A, B, offset_h=0, tol_s=10.0, name_A='A', name_B='B'):
    """
    For each event in A, find the nearest event in B within tol_s seconds
    after adding offset_h hours to A's times.

    Returns a DataFrame of matched pairs with spatial and magnitude differences.
    """
    B_ms  = B.time.values.astype('datetime64[ms]').astype(np.int64)
    off_ms = int(offset_h * 3600 * 1000)
    tol_ms = int(tol_s * 1000)
    pairs  = []
    for _, a in A.iterrows():
        t_a  = int(pd.Timestamp(a.time).value // 1e6) + off_ms
        diffs = np.abs(B_ms - t_a)
        bi    = int(np.argmin(diffs))
        if diffs[bi] > tol_ms:
            continue
        b = B.iloc[bi]
        dt_s  = (t_a - int(pd.Timestamp(b.time).value // 1e6)) / 1000.0
        dlat  = a.lat - b.lat
        dlon  = a.lon - b.lon
        dlat_km = dlat * 111.0
        dlon_km = dlon * 111.0 * np.cos(np.radians(a.lat))
        dist_km = np.sqrt(dlat_km**2 + dlon_km**2)
        pairs.append({
            'dt_s': dt_s,
            'dist_km': dist_km,
            'ddep': a.dep - b.dep,
            'dmag': a.mag - b.mag,
            f'{name_A}_lat': a.lat, f'{name_A}_lon': a.lon,
            f'{name_A}_dep': a.dep, f'{name_A}_mag': a.mag,
            f'{name_B}_lat': b.lat, f'{name_B}_lon': b.lon,
            f'{name_B}_dep': b.dep, f'{name_B}_mag': b.mag,
            f'{name_B}_time': b.time,
        })
    df = pd.DataFrame(pairs)
    n, N = len(df), len(A)
    print(f"  {name_A} vs {name_B} (offset={offset_h:+g}h, tol={tol_s}s): "
          f"{n}/{N} matched ({100*n/N:.0f}%)")
    if len(df):
        vd = df.dropna(subset=['ddep'])
        vm = df.dropna(subset=['dmag'])
        print(f"    dist   med={df.dist_km.median():.2f} km  "
              f"90th={np.percentile(df.dist_km,90):.2f} km")
        print(f"    ddep   med={vd.ddep.median():.2f} km  "
              f"σ={vd.ddep.std():.2f} km  (n={len(vd)})")
        print(f"    dmag   med={vm.dmag.median():.2f}  "
              f"σ={vm.dmag.std():.2f}  (n={len(vm)})")
    return df


print("Matching catalogs...")
# MORI vs PINAALL — same processing, should be near-identical for shared events
m_mp  = match_catalogs(pina, mori,    0, 5.0, 'PINAALL', 'MORI')
# MORI (May–Jun) vs BRUN with +7 h
m_mb7 = match_catalogs(mori_mj, brun, 7, 5.0, 'MORI_MJ', 'BRUN')
# MORI (Jul–Aug) vs BRUN no offset
m_mb0 = match_catalogs(mori_ja, brun, 0, 5.0, 'MORI_JA', 'BRUN')
# GOLD vs BRUN with +7 h
m_gb  = match_catalogs(gold, brun,    7, 10.0, 'GOLD',   'BRUN')

## 5  Figures

### 5.1  Three-panel epicentre map

In [ ]:
def _map_panel(ax, A_lon, A_lat, B_lon, B_lat, title,
               color_A='coral', color_B='steelblue',
               label_A='Catalog A', label_B='Catalog B'):
    ax.scatter(B_lon, B_lat, s=3, alpha=0.25, c=color_B, label=label_B, zorder=2)
    ax.scatter(A_lon, A_lat, s=3, alpha=0.25, c=color_A,
               label=label_A, marker='^', zorder=3)
    ax.plot(*PINATUBO, 'k^', ms=9, zorder=6, label='Summit')
    ax.set_xlim(MAP_XLIM)
    ax.set_ylim(MAP_YLIM)
    ax.set_xlabel('Longitude (°E)')
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=7, markerscale=3, loc='upper left')
    ax.grid(True, alpha=0.25)


fig, axes = plt.subplots(1, 3, figsize=(15, 5.5), sharey=True)
fig.suptitle(
    'Epicentres of common events across the recovered hypocenter catalogs\n'
    '(GOLD and MORI May–Jun times shifted +7 h before matching; tolerance 5–10 s)',
    fontsize=10)

_map_panel(axes[0],
           m_mp['PINAALL_lon'], m_mp['PINAALL_lat'],
           m_mp['MORI_lat'].rename('x'), m_mp['MORI_lon'],  # swap for clarity
           f'PINAALL vs MORI\n(no offset; n = {len(m_mp)})',
           color_A='mediumpurple', color_B='steelblue',
           label_A='PINAALL (HYPO71)', label_B='MORI (HYPO71)')
# Fix: use correct columns
_map_panel(axes[0],
           m_mp['PINAALL_lon'], m_mp['PINAALL_lat'],
           m_mp['MORI_lon'], m_mp['MORI_lat'],
           f'PINAALL vs MORI (no offset)\nn = {len(m_mp)}',
           color_A='mediumpurple', color_B='steelblue',
           label_A='PINAALL (HYPO71)', label_B='MORI (HYPO71)')

_map_panel(axes[1],
           m_mb7['MORI_MJ_lon'], m_mb7['MORI_MJ_lat'],
           m_mb7['BRUN_lon'],    m_mb7['BRUN_lat'],
           f'MORI (May–Jun) vs BRUN (+7 h)\nn = {len(m_mb7)}',
           color_A='coral', color_B='steelblue',
           label_A='MORI (HYPO71)', label_B='BRUN (HYPOELLIPSE 2001)')

_map_panel(axes[2],
           m_gb['GOLD_lon'], m_gb['GOLD_lat'],
           m_gb['BRUN_lon'], m_gb['BRUN_lat'],
           f'GOLD vs BRUN (+7 h)\nn = {len(m_gb)}',
           color_A='darkorange', color_B='steelblue',
           label_A='GOLD (HYPOELLIPSE 1992–93)', label_B='BRUN (HYPOELLIPSE 2001)')

axes[0].set_ylabel('Latitude (°N)')
plt.tight_layout()
for ext in ('pdf', 'png'):
    plt.savefig(FIG_DIR / f'catalog_maps.{ext}', bbox_inches='tight')
plt.show()
print("Saved catalog_maps.pdf/png")

### 5.2  Comparison statistics (distance, depth difference, magnitude difference)

In [ ]:
def _stats_row(axes_row, m, label_A, label_B):
    vd = m.dropna(subset=['ddep'])
    vm = m.dropna(subset=['dmag'])
    # Horizontal distance
    ax = axes_row[0]
    ax.hist(m.dist_km.clip(0, 10), bins=40, color='steelblue',
            edgecolor='k', linewidth=0.3)
    ax.axvline(m.dist_km.median(), color='r', ls='--', lw=1.5,
               label=f'median {m.dist_km.median():.2f} km')
    ax.set_xlabel('Epicentral separation (km)')
    ax.set_ylabel('Count')
    ax.set_title(f'{label_A} vs {label_B}\nHorizontal distance  n={len(m)}')
    ax.legend(fontsize=8)
    # Depth difference
    ax = axes_row[1]
    ax.hist(vd.ddep.clip(-10, 10), bins=40, color='forestgreen',
            edgecolor='k', linewidth=0.3)
    ax.axvline(0, color='r', ls='--', lw=1.5)
    ax.axvline(vd.ddep.median(), color='orange', ls=':', lw=1.5,
               label=f'median {vd.ddep.median():.2f} km')
    ax.set_xlabel(f'{label_A} − {label_B} depth (km)')
    ax.set_ylabel('Count')
    ax.set_title(f'Depth difference  σ={vd.ddep.std():.2f} km  n={len(vd)}')
    ax.legend(fontsize=8)
    # Magnitude difference
    ax = axes_row[2]
    ax.hist(vm.dmag.dropna().clip(-3, 3), bins=40, color='coral',
            edgecolor='k', linewidth=0.3)
    ax.axvline(0, color='r', ls='--', lw=1.5)
    ax.axvline(float(vm.dmag.median()), color='orange', ls=':', lw=1.5,
               label=f'median {float(vm.dmag.median()):.2f}')
    ax.set_xlabel(f'{label_A} − {label_B} magnitude')
    ax.set_ylabel('Count')
    ax.set_title(f'Magnitude difference  σ={vm.dmag.std():.2f}  n={len(vm)}')
    ax.legend(fontsize=8)


fig2, axes2 = plt.subplots(3, 3, figsize=(14, 11))
fig2.suptitle('Comparison statistics for matched event pairs', fontsize=12)
_stats_row(axes2[0], m_mp,  'PINAALL', 'MORI')
_stats_row(axes2[1], m_mb0, 'MORI (Jul–Aug)', 'BRUN')
_stats_row(axes2[2], m_gb,  'GOLD', 'BRUN')
plt.tight_layout()
for ext in ('pdf', 'png'):
    plt.savefig(FIG_DIR / f'catalog_stats.{ext}', bbox_inches='tight')
plt.show()
print("Saved catalog_stats.pdf/png")

### 5.3  Processing-date histogram (GOLD.SUM — provenance evidence)

In [ ]:
gold_valid = gold.dropna(subset=['proc_month', 'proc_year'])
if len(gold_valid):
    # Create approximate processing dates (year 1900+py, month pm)
    gold_valid = gold_valid.copy()
    gold_valid['proc_yr4'] = (gold_valid.proc_year + 1900).astype(int)
    gold_valid['proc_label'] = gold_valid.apply(
        lambda r: f"{int(r.proc_yr4)}-{int(r.proc_month):02d}", axis=1)
    counts = gold_valid.proc_label.value_counts().sort_index()

    fig3, ax = plt.subplots(figsize=(9, 4))
    ax.bar(range(len(counts)), counts.values, color='steelblue', edgecolor='k', lw=0.5)
    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels(counts.index, rotation=45, ha='right', fontsize=8)
    ax.set_xlabel('Processing month (YYYY-MM) recorded in HYPOELLIPSE header')
    ax.set_ylabel('Number of events')
    ax.set_title('GOLD.SUM — processing dates (July 1992 – April 1993)\n'
                 'Evidence of retrospective reprocessing ~1 year after the eruption')
    plt.tight_layout()
    for ext in ('pdf', 'png'):
        plt.savefig(FIG_DIR / f'gold_processing_dates.{ext}', bbox_inches='tight')
    plt.show()
    print("Saved gold_processing_dates.pdf/png")

## 6  Summary table for paper

Numerical values to transcribe into Table 1 of the manuscript.

In [ ]:
print("=" * 70)
print("NUMBERS FOR PAPER / SUPPLEMENT")
print("=" * 70)

pairs = [
    ('PINAALL vs MORI (0 h)',    m_mp,  'PINAALL', 'MORI',    len(pina),  len(mori)),
    ('MORI(May–Jun) vs BRUN (+7h)', m_mb7, 'MORI_MJ', 'BRUN', len(mori_mj), len(brun)),
    ('MORI(Jul–Aug) vs BRUN (0h)', m_mb0, 'MORI_JA', 'BRUN',  len(mori_ja), len(brun)),
    ('GOLD vs BRUN (+7h)',        m_gb,  'GOLD',    'BRUN',    len(gold),  len(brun)),
]
for label, m, nA_col, nB_col, nA, nB in pairs:
    if m.empty:
        continue
    vd = m.dropna(subset=['ddep'])
    vm = m.dropna(subset=['dmag'])
    print(f"\n{label}:  {len(m)} / {nA} matched ({100*len(m)/nA:.0f}%)")
    print(f"  Horizontal:  med={m.dist_km.median():.2f} km  "
          f"90th={np.percentile(m.dist_km,90):.2f} km  "
          f"<1km: {100*(m.dist_km<1).mean():.0f}%  "
          f"<5km: {100*(m.dist_km<5).mean():.0f}%")
    print(f"  Depth diff:  med={vd.ddep.median():.2f} km  "
          f"σ={vd.ddep.std():.2f} km  "
          f"|Δ|<2km: {100*(np.abs(vd.ddep)<2).mean():.0f}%  (n={len(vd)})")
    print(f"  Mag diff:    med={float(vm.dmag.median()):.2f}  "
          f"σ={vm.dmag.std():.2f}  (n={len(vm)})")